# Part 2: Beyond CIFAR-10 — Food-101 Transfer Learning

ECS7026P Neural Networks and Deep Learning  

**Goal**: Fine-tune ImageNet-pretrained models on Food-101 and run reproducible experiments.

**Models**: ResNet-18 (baseline) and EfficientNet-B0 (main).  
**Dataset**: Food-101 (101 classes).

## 0. Colab notes

- Runtime → Change runtime type → **GPU (T4)**
- This notebook uses `torchvision.datasets.Food101`.
- All experiments are controlled by the config dictionary in Section 6.


## 1. Setup

This section sets seeds, selects the device, and defines a few helpers for reproducibility and timing.

In [ ]:
import os
import time
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


seed_everything(42)
device = get_device()
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("device:", device)
print("cuda name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Data (Food-101)

We use the official Food-101 train/test split provided by `torchvision`. We also carve out a validation split from the training set for model selection.

**Input resolution**: 224×224 for both models (standard ImageNet size).

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def make_transforms(img_size: int = 224, strong_aug: bool = False):
    if not strong_aug:
        train_tf = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:
        # Stronger aug: RandAugment + RandomErasing. (MixUp/CutMix handled in the training loop.)
        train_tf = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandAugment(num_ops=2, magnitude=9),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            T.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), value="random"),
        ])

    eval_tf = T.Compose([
        T.Resize(int(img_size * 256 / 224)),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    return train_tf, eval_tf


def split_indices(n: int, val_frac: float, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_val = int(round(n * val_frac))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]
    return tr_idx, val_idx


def make_dataloaders(
    data_root: str = "./data",
    batch_size: int = 64,
    num_workers: int = 2,
    img_size: int = 224,
    strong_aug: bool = False,
    val_frac: float = 0.1,
    seed: int = 42,
):
    train_tf, eval_tf = make_transforms(img_size=img_size, strong_aug=strong_aug)

    train_full = torchvision.datasets.Food101(root=data_root, split="train", download=True, transform=train_tf)
    test_ds = torchvision.datasets.Food101(root=data_root, split="test", download=True, transform=eval_tf)

    tr_idx, val_idx = split_indices(len(train_full), val_frac=val_frac, seed=seed)

    train_ds = Subset(train_full, tr_idx)

    # For validation, we need eval transforms (no augmentation)
    val_base = torchvision.datasets.Food101(root=data_root, split="train", download=False, transform=eval_tf)
    val_ds = Subset(val_base, val_idx)

    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    return train_loader, val_loader, test_loader


# Quick sanity check
train_loader, val_loader, test_loader = make_dataloaders(batch_size=32, strong_aug=False, img_size=224)
X, y = next(iter(train_loader))
print("train batch:", X.shape, y.shape)
print("num classes:", 101)

## 3. Models (pretrained)

We use ImageNet-pretrained weights and replace the classifier head for 101 classes.

We also support a **fine-tuning depth** ablation:
- `"head"`: train only the classifier head
- `"last_block"`: unfreeze the last stage + head
- `"all"`: unfreeze the whole network

In [ ]:
def set_requires_grad(module: nn.Module, requires_grad: bool) -> None:
    for p in module.parameters():
        p.requires_grad = requires_grad


def build_model(model_name: str, num_classes: int = 101, finetune: str = "all") -> nn.Module:
    model_name = model_name.lower()

    if model_name == "resnet18":
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        model = torchvision.models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

        if finetune == "head":
            set_requires_grad(model, False)
            set_requires_grad(model.fc, True)
        elif finetune == "last_block":
            set_requires_grad(model, False)
            # layer4 + head
            set_requires_grad(model.layer4, True)
            set_requires_grad(model.fc, True)
        elif finetune == "all":
            set_requires_grad(model, True)
        else:
            raise ValueError(f"Unknown finetune mode: {finetune}")

        return model

    if model_name == "efficientnet_b0":
        weights = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = torchvision.models.efficientnet_b0(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)

        if finetune == "head":
            set_requires_grad(model, False)
            set_requires_grad(model.classifier, True)
        elif finetune == "last_block":
            set_requires_grad(model, False)
            # Unfreeze last feature block + classifier
            set_requires_grad(model.features[-1], True)
            set_requires_grad(model.classifier, True)
        elif finetune == "all":
            set_requires_grad(model, True)
        else:
            raise ValueError(f"Unknown finetune mode: {finetune}")

        return model

    raise ValueError(f"Unknown model: {model_name}")


# Sanity check
m = build_model("resnet18", finetune="head").to(device)
print("resnet18 params trainable:", sum(p.requires_grad for p in m.parameters()), "/", sum(1 for _ in m.parameters()))

del m
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Training + evaluation utilities

- Mixed precision (`torch.cuda.amp`) for speed on T4
- Optional MixUp/CutMix for the strong-augmentation setting
- Logs: loss, top-1 acc, top-5 acc, epoch time

In [ ]:
@torch.no_grad()
def accuracy_topk(logits: torch.Tensor, targets: torch.Tensor, ks=(1, 5)) -> Dict[str, float]:
    maxk = max(ks)
    _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)
    pred = pred.t()  # [K, B]
    correct = pred.eq(targets.view(1, -1).expand_as(pred))

    out = {}
    for k in ks:
        corr_k = correct[:k].reshape(-1).float().sum().item()
        out[f"top{k}"] = corr_k / targets.size(0)
    return out


def one_hot(y: torch.Tensor, num_classes: int) -> torch.Tensor:
    return F.one_hot(y, num_classes=num_classes).float()


def mixup_cutmix(
    x: torch.Tensor,
    y: torch.Tensor,
    num_classes: int,
    mixup_alpha: float = 0.2,
    cutmix_alpha: float = 1.0,
    p: float = 0.5,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Returns (x_mixed, y_soft). y_soft is a probability distribution."""
    if (mixup_alpha <= 0 and cutmix_alpha <= 0) or random.random() > p:
        return x, one_hot(y, num_classes)

    use_cutmix = (cutmix_alpha > 0) and (mixup_alpha <= 0 or random.random() < 0.5)

    if not use_cutmix:
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        idx = torch.randperm(x.size(0), device=x.device)
        x2 = x[idx]
        y2 = y[idx]
        x_m = lam * x + (1 - lam) * x2
        y_m = lam * one_hot(y, num_classes) + (1 - lam) * one_hot(y2, num_classes)
        return x_m, y_m

    # CutMix
    lam = np.random.beta(cutmix_alpha, cutmix_alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x2 = x[idx]
    y2 = y[idx]

    B, C, H, W = x.shape
    cut_rat = math.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2b = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2b = np.clip(cy + cut_h // 2, 0, H)

    x_m = x.clone()
    x_m[:, :, y1:y2b, x1:x2b] = x2[:, :, y1:y2b, x1:x2b]

    # adjust lambda based on actual area
    lam_adj = 1.0 - ((x2b - x1) * (y2b - y1) / (H * W))
    y_m = lam_adj * one_hot(y, num_classes) + (1 - lam_adj) * one_hot(y2, num_classes)
    return x_m, y_m


def soft_cross_entropy(logits: torch.Tensor, y_soft: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=1)
    return -(y_soft * logp).sum(dim=1).mean()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    n = 0

    ce = nn.CrossEntropyLoss()

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = ce(logits, y)
        acc = accuracy_topk(logits, y, ks=(1, 5))

        bsz = y.size(0)
        total_loss += loss.item() * bsz
        total_top1 += acc["top1"] * bsz
        total_top5 += acc["top5"] * bsz
        n += bsz

    return {
        "loss": total_loss / n,
        "top1": total_top1 / n,
        "top5": total_top5 / n,
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    scaler: torch.cuda.amp.GradScaler,
    num_classes: int = 101,
    label_smoothing: float = 0.0,
    use_mix: bool = False,
) -> Dict[str, float]:
    model.train()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    n = 0

    ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_mix:
            x, y_soft = mixup_cutmix(x, y, num_classes=num_classes, p=0.5)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            if use_mix:
                loss = soft_cross_entropy(logits, y_soft)
                # for logging, compute acc against hard labels
                y_hard = y
            else:
                loss = ce(logits, y)
                y_hard = y

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            acc = accuracy_topk(logits.detach(), y_hard, ks=(1, 5))

        bsz = y.size(0)
        total_loss += loss.item() * bsz
        total_top1 += acc["top1"] * bsz
        total_top5 += acc["top5"] * bsz
        n += bsz

    return {
        "loss": total_loss / n,
        "top1": total_top1 / n,
        "top5": total_top5 / n,
    }

## 5. Single run function

This wraps: dataloaders → model → optimizer/scheduler → train/val loop → best-checkpoint selection → final test evaluation.

It returns a dict you can append into a results table.

In [ ]:
def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def run_experiment(
    *,
    model_name: str,
    finetune: str,
    strong_aug: bool,
    use_mix: bool,
    epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
    label_smoothing: float,
    img_size: int = 224,
    data_root: str = "./data",
    num_workers: int = 2,
    seed: int = 42,
    save_dir: str = "./runs_food101",
) -> Dict[str, float]:
    seed_everything(seed)
    os.makedirs(save_dir, exist_ok=True)

    train_loader, val_loader, test_loader = make_dataloaders(
        data_root=data_root,
        batch_size=batch_size,
        num_workers=num_workers,
        img_size=img_size,
        strong_aug=strong_aug,
        val_frac=0.1,
        seed=seed,
    )

    model = build_model(model_name, num_classes=101, finetune=finetune).to(device)

    # Only optimize trainable parameters
    params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    run_id = f"{model_name}__ft={finetune}__aug={'strong' if strong_aug else 'basic'}__mix={int(use_mix)}__lr={lr}__seed={seed}"
    ckpt_path = os.path.join(save_dir, run_id + ".pt")

    best_val_top1 = -1.0
    best_epoch = -1
    history = []

    start_all = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device,
            scaler,
            num_classes=101,
            label_smoothing=label_smoothing,
            use_mix=(use_mix and strong_aug),
        )
        val = evaluate(model, val_loader, device)
        scheduler.step()
        epoch_s = time.time() - t0

        row = {
            "epoch": epoch,
            "train_loss": tr["loss"],
            "train_top1": tr["top1"],
            "val_loss": val["loss"],
            "val_top1": val["top1"],
            "val_top5": val["top5"],
            "epoch_s": epoch_s,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        if val["top1"] > best_val_top1:
            best_val_top1 = val["top1"]
            best_epoch = epoch
            torch.save(
                {
                    "model_name": model_name,
                    "finetune": finetune,
                    "strong_aug": strong_aug,
                    "use_mix": use_mix,
                    "img_size": img_size,
                    "state_dict": model.state_dict(),
                    "epoch": epoch,
                    "val": val,
                },
                ckpt_path,
            )

        print(
            f"[{run_id}] epoch {epoch:02d}/{epochs} | "
            f"tr top1 {tr['top1']*100:5.2f}% loss {tr['loss']:.3f} | "
            f"val top1 {val['top1']*100:5.2f}% top5 {val['top5']*100:5.2f}% loss {val['loss']:.3f} | "
            f"{epoch_s:.1f}s"
        )

    total_s = time.time() - start_all

    # Load best checkpoint, evaluate on test
    best = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(best["state_dict"])
    test = evaluate(model, test_loader, device)

    result = {
        "run_id": run_id,
        "model": model_name,
        "finetune": finetune,
        "aug": "strong" if strong_aug else "basic",
        "mix": int(use_mix and strong_aug),
        "epochs": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "weight_decay": weight_decay,
        "label_smoothing": label_smoothing,
        "seed": seed,
        "trainable_params": count_trainable_params(model),
        "best_epoch": best_epoch,
        "best_val_top1": best_val_top1,
        "test_loss": test["loss"],
        "test_top1": test["top1"],
        "test_top5": test["top5"],
        "total_time_s": total_s,
        "ckpt_path": ckpt_path,
        "history": history,
    }

    return result

## 6. Experiment grid (Part 2)

Suggested experiments for your report:

- **Model comparison**: ResNet-18 vs EfficientNet-B0
- **Fine-tuning depth**: `head` vs `last_block` vs `all`
- **Augmentation**: `basic` vs `strong` (RandAugment + optional MixUp/CutMix)

Tip: start with the **quick grid** (few epochs) to find sensible hyperparameters, then rerun the best configs for longer.

In [ ]:
import pandas as pd

# ---- Config you will edit ----
CFG = {
    "data_root": "./data",
    "img_size": 224,
    "num_workers": 2,
    "save_dir": "./runs_food101",

    # Quick vs full runs
    "quick_epochs": 2,
    "full_epochs": 8,

    # Defaults that fit on a T4; reduce batch_size if you OOM
    "batch_size": 64,

    # Optimizer
    "lr": 3e-4,
    "weight_decay": 1e-2,
    "label_smoothing": 0.1,

    # Repro
    "seed": 42,
}

# Quick experiment grid (fast signal)
QUICK_GRID = [
    # Baseline: ResNet18
    {"model_name": "resnet18", "finetune": "head", "strong_aug": False, "use_mix": False},
    {"model_name": "resnet18", "finetune": "last_block", "strong_aug": False, "use_mix": False},
    {"model_name": "resnet18", "finetune": "all", "strong_aug": False, "use_mix": False},

    # Main: EfficientNet-B0
    {"model_name": "efficientnet_b0", "finetune": "head", "strong_aug": False, "use_mix": False},
    {"model_name": "efficientnet_b0", "finetune": "last_block", "strong_aug": False, "use_mix": False},
    {"model_name": "efficientnet_b0", "finetune": "all", "strong_aug": False, "use_mix": False},

    # Augmentation ablation (strong aug)
    {"model_name": "resnet18", "finetune": "all", "strong_aug": True, "use_mix": True},
    {"model_name": "efficientnet_b0", "finetune": "all", "strong_aug": True, "use_mix": True},
]


def run_grid(grid, epochs: int):
    results = []
    for g in grid:
        out = run_experiment(
            **g,
            epochs=epochs,
            batch_size=CFG["batch_size"],
            lr=CFG["lr"],
            weight_decay=CFG["weight_decay"],
            label_smoothing=CFG["label_smoothing"],
            img_size=CFG["img_size"],
            data_root=CFG["data_root"],
            num_workers=CFG["num_workers"],
            seed=CFG["seed"],
            save_dir=CFG["save_dir"],
        )
        results.append(out)
    return results


# ---- Run quick grid ----
quick_results = run_grid(QUICK_GRID, epochs=CFG["quick_epochs"])

# Make a table
rows = []
for r in quick_results:
    rows.append({k: r[k] for k in [
        "run_id","model","finetune","aug","mix","epochs","batch_size","lr","weight_decay","label_smoothing",
        "trainable_params","best_epoch","best_val_top1","test_top1","test_top5","total_time_s"
    ]})

df = pd.DataFrame(rows).sort_values(["test_top1"], ascending=False)
df

## 7. Plots + export

This produces learning curves for the best run and exports a CSV you can include in your report appendix.

In [ ]:
# Save results table
os.makedirs(CFG["save_dir"], exist_ok=True)
out_csv = os.path.join(CFG["save_dir"], "quick_grid_results.csv")
df.to_csv(out_csv, index=False)
print("saved:", out_csv)

# Plot learning curves for best run
best_run_id = df.iloc[0]["run_id"]
best = next(r for r in quick_results if r["run_id"] == best_run_id)
hist = pd.DataFrame(best["history"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["epoch"], hist["train_loss"], label="train")
ax[0].plot(hist["epoch"], hist["val_loss"], label="val")
ax[0].set_title("Loss")
ax[0].set_xlabel("epoch")
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(hist["epoch"], 100*hist["train_top1"], label="train")
ax[1].plot(hist["epoch"], 100*hist["val_top1"], label="val")
ax[1].set_title("Top-1 accuracy (%)")
ax[1].set_xlabel("epoch")
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.suptitle(best_run_id)
plt.show()

print("Best run summary:")
print({k: best[k] for k in ["model","finetune","aug","mix","best_epoch","best_val_top1","test_top1","test_top5","total_time_s"]})